[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/06_xgboost_regressor/XGBoost_Regressor.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/06_xgboost_regressor/XGBoost_Regressor.ipynb)

# Notebook 06 — XGBoost Regressor
## Gradient Boosting + Full Method Comparison

**Dataset**: California Housing
**Question**: *What is the median house value — and how do all six methods compare?*
**Type**: Supervised Regression — Gradient Boosting

---

## Section 1 — What Is XGBoost Regressor?

### In plain English

Random Forest asks 100 surveyors all at once and averages their estimates.
**XGBoost asks them one at a time — each surveyor fixes the previous one's mistakes.**

> **Round 1**: Surveyor 1 predicts values. Some are too high, some too low.
> **Round 2**: Surveyor 2 focuses on the districts where Surveyor 1 was furthest off.
> **Round 3**: Surveyor 3 focuses on the remaining errors.
> ... repeat ...
> **Final estimate**: sum of all 100 surveyors' corrections.

Each "surveyor" is a small, shallow regression tree.
Each tree predicts the **residual error** left by all previous trees.

### For regression, the residual is literally the error

```
Round 1: predict 2.1,  actual 3.0  → residual = +0.9
Round 2: predict 0.6 for the residual  → new total = 2.1 + 0.6 = 2.7
Round 3: predict 0.2 for remaining residual  → new total = 2.7 + 0.2 = 2.9
...
```

## Section 2 — How Does It Learn?

**Boosting in 4 steps:**

1. **Start**: predict the training mean for all districts
2. **Compute residuals**: `residual = actual − current_prediction`
3. **Fit a small tree to the residuals**: tree learns "where am I most wrong, by how much?"
4. **Update**: `new_prediction = old_prediction + learning_rate × tree_correction`

The learning rate (e.g. 0.1) keeps each tree's contribution small,
preventing any single tree from overcorrecting.

**XGBoost adds regularisation on top of standard gradient boosting:**
- L1 and L2 penalties on tree weights (like Lasso/Ridge but for tree leaf values)
- Tree structure regularisation (penalises too many leaves)
- This makes it more robust and less likely to overfit than standard gradient boosting.

## Section 3 — What Does the Data Need?

Same as all tree-based methods: no scaling required.
XGBoost also handles missing values natively (not demonstrated here since
California Housing has no missing values).

| Feature | Transform |
|---|---|
| All features | None — raw values work |
| AveRooms / AveBedrms / AveOccup | Clip outliers (already done) |

In [ ]:
try:
    import xgboost as xgb
    print(f"XGBoost {xgb.__version__} ready")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    import xgboost as xgb
    print(f"Installed XGBoost {xgb.__version__}")

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target
for col in ['AveRooms', 'AveBedrms', 'AveOccup']:
    df[col] = df[col].clip(upper=df[col].quantile(0.99))

FEATURES = housing.feature_names.tolist()
TARGET   = 'MedHouseVal'

print(f"California Housing: {df.shape[0]} districts, {len(FEATURES)} features")
print(f"Target (MedHouseVal): median house value in $100k, range {df[TARGET].min():.2f}–{df[TARGET].max():.2f}")
df.head(6)

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train: {len(X_train)} districts  |  Test: {len(X_test)} districts")
print("No scaling applied — tree-based models do not need it.")

## Section 4 — Train XGBoost and Compare All Six Methods

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5,
                          random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)
y_pred  = xgb_model.predict(X_test)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred))
r2_xgb   = r2_score(y_test, y_pred)

print(f"XGBoost Regressor: RMSE={rmse_xgb:.4f}  R²={r2_xgb:.4f}")

In [ ]:
# Re-train all methods for fair comparison
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

X_all = df[FEATURES].values
y_all = df[TARGET].values
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr)
X_te_sc = sc.transform(X_te)

methods = {}
methods['Linear Regression'] = LinearRegression().fit(X_tr_sc, y_tr)
methods['Ridge Regression']  = RidgeCV(alphas=np.logspace(-2, 4, 50), cv=5).fit(X_tr_sc, y_tr)
methods['Lasso Regression']  = LassoCV(alphas=np.logspace(-4, 1, 50), cv=5, max_iter=5000).fit(X_tr_sc, y_tr)
methods['Decision Tree']     = DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_tr, y_tr)
methods['Random Forest']     = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
methods['XGBoost']           = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5,
                                             random_state=42, verbosity=0).fit(X_tr, y_tr)

rows = []
for name, m in methods.items():
    X_eval = X_te_sc if name in ('Linear Regression','Ridge Regression','Lasso Regression') else X_te
    y_p = m.predict(X_eval)
    rows.append({
        'Method'         : name,
        'RMSE'           : round(np.sqrt(mean_squared_error(y_te, y_p)), 4),
        'R²'             : round(r2_score(y_te, y_p), 4),
        'Needs Scaling'  : 'Yes' if name in ('Linear Regression','Ridge Regression','Lasso Regression') else 'No',
    })

results_df = pd.DataFrame(rows)
print("=== All Six Regression Methods on California Housing ===")
print(results_df.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

colors = ['#5C6BC0','#7986CB','#9FA8DA','#FFA726','#26A69A','#AB47BC']
bars = ax1.bar(results_df['Method'], results_df['RMSE'], color=colors, width=0.6)
ax1.set_ylabel('Test RMSE (lower = better)', fontsize=11)
ax1.set_title('RMSE Comparison', fontsize=12)
ax1.tick_params(axis='x', rotation=20)
for bar, v in zip(bars, results_df['RMSE']):
    ax1.text(bar.get_x()+bar.get_width()/2, v+0.002, f'{v:.3f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

bars2 = ax2.bar(results_df['Method'], results_df['R²'], color=colors, width=0.6)
ax2.set_ylim(0, 1)
ax2.set_ylabel('R² (higher = better)', fontsize=11)
ax2.set_title('R² Comparison', fontsize=12)
ax2.tick_params(axis='x', rotation=20)
for bar, v in zip(bars2, results_df['R²']):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.005, f'{v:.3f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print()
print('Key takeaways:')
print('  1. Linear/Ridge/Lasso capture linear patterns — all similar on this dataset')
print('  2. Lasso automatically zeroes out weak features (interpretability advantage)')
print('  3. Tree methods capture non-linear patterns — much higher R²')
print('  4. XGBoost typically leads in accuracy but requires more tuning')

In [ ]:
# XGBoost feature importances
importance_df = pd.DataFrame({
    'Feature': FEATURES, 'Importance': xgb_model.feature_importances_.round(4)
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance_df['Feature'], importance_df['Importance'], color='#AB47BC')
ax.set_title('XGBoost Feature Importances\n(contribution across all 200 boosting rounds)', fontsize=12)
ax.set_xlabel('Importance score', fontsize=11)
plt.tight_layout()
plt.show()